# Heart Disease Prediction

This notebook demonstrates a machine learning model to predict heart disease based on patient data.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 2. Load Dataset

We'll use the UCI Heart Disease dataset which is commonly available.

In [ ]:
# Load the heart disease dataset
# You can download from: https://archive.ics.uci.edu/ml/datasets/Heart+Disease
# Or use the dataset from Kaggle or other sources

# For demonstration, we'll create a sample loading code
# Replace 'heart.csv' with your actual dataset path
try:
    df = pd.read_csv('heart.csv')
    print("Dataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
except FileNotFoundError:
    print("Dataset file not found. Please download the heart disease dataset.")
    print("Creating a sample dataset for demonstration...")
    # Create a sample dataset
    n_samples = 300
    df = pd.DataFrame({
        'age': np.random.randint(30, 80, n_samples),
        'sex': np.random.randint(0, 2, n_samples),
        'cp': np.random.randint(0, 4, n_samples),
        'trestbps': np.random.randint(90, 200, n_samples),
        'chol': np.random.randint(120, 400, n_samples),
        'fbs': np.random.randint(0, 2, n_samples),
        'restecg': np.random.randint(0, 3, n_samples),
        'thalach': np.random.randint(70, 200, n_samples),
        'exang': np.random.randint(0, 2, n_samples),
        'oldpeak': np.random.uniform(0, 6, n_samples),
        'slope': np.random.randint(0, 3, n_samples),
        'ca': np.random.randint(0, 4, n_samples),
        'thal': np.random.randint(0, 4, n_samples),
        'target': np.random.randint(0, 2, n_samples)
    })

## 3. Data Exploration

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Dataset information
print("Dataset Information:")
df.info()

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("Missing values in dataset:")
print(df.isnull().sum())

In [ ]:
# Check target distribution
print("Target variable distribution:")
print(df['target'].value_counts())

# Visualize target distribution
plt.figure(figsize=(8, 6))
sns.countplot(x='target', data=df)
plt.title('Distribution of Target Variable')
plt.xlabel('Heart Disease (0: No, 1: Yes)')
plt.ylabel('Count')
plt.show()

## 4. Data Visualization

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Age distribution by target
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='age', hue='target', bins=20, kde=True)
plt.title('Age Distribution by Heart Disease Status')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

## 5. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")

## 6. Model Training

### 6.1 Logistic Regression

In [ ]:
# Train Logistic Regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the model
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_pred_proba_lr)

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")
print(f"Logistic Regression AUC-ROC: {lr_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

### 6.2 Random Forest Classifier

In [ ]:
# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the model
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_pred_proba_rf)

print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest AUC-ROC: {rf_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

## 7. Model Evaluation

In [ ]:
# Confusion Matrix for Logistic Regression
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')

# Confusion Matrix for Random Forest
plt.subplot(1, 2, 2)
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix - Random Forest')
plt.ylabel('Actual')
plt.xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
plt.figure(figsize=(10, 6))

# Logistic Regression ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_auc:.4f})')

# Random Forest ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {rf_auc:.4f})')

# Random classifier line
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Feature Importance (Random Forest)
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance)
plt.title('Feature Importance - Random Forest')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print("Feature Importance:")
print(feature_importance)

## 8. Model Comparison

In [ ]:
# Compare model performance
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [lr_accuracy, rf_accuracy],
    'AUC-ROC': [lr_auc, rf_auc]
})

print("Model Performance Comparison:")
print(comparison_df)

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].bar(comparison_df['Model'], comparison_df['Accuracy'], color=['blue', 'green'])
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0, 1])
for i, v in enumerate(comparison_df['Accuracy']):
    axes[0].text(i, v + 0.02, f'{v:.4f}', ha='center')

# AUC-ROC comparison
axes[1].bar(comparison_df['Model'], comparison_df['AUC-ROC'], color=['blue', 'green'])
axes[1].set_title('Model AUC-ROC Comparison')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_ylim([0, 1])
for i, v in enumerate(comparison_df['AUC-ROC']):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center')

plt.tight_layout()
plt.show()

## 9. Making Predictions on New Data

In [ ]:
# Example: Make prediction for a new patient
# Create a sample patient data (replace with actual values)
new_patient = pd.DataFrame({
    'age': [55],
    'sex': [1],
    'cp': [2],
    'trestbps': [140],
    'chol': [250],
    'fbs': [1],
    'restecg': [0],
    'thalach': [150],
    'exang': [0],
    'oldpeak': [2.5],
    'slope': [1],
    'ca': [1],
    'thal': [2]
})

# Scale the new patient data
new_patient_scaled = scaler.transform(new_patient)

# Make prediction using both models
lr_prediction = lr_model.predict(new_patient_scaled)[0]
lr_probability = lr_model.predict_proba(new_patient_scaled)[0]

rf_prediction = rf_model.predict(new_patient_scaled)[0]
rf_probability = rf_model.predict_proba(new_patient_scaled)[0]

print("Prediction for New Patient:")
print("="*50)
print(f"Logistic Regression Prediction: {'Heart Disease' if lr_prediction == 1 else 'No Heart Disease'}")
print(f"Probability: No Disease={lr_probability[0]:.4f}, Disease={lr_probability[1]:.4f}")
print()
print(f"Random Forest Prediction: {'Heart Disease' if rf_prediction == 1 else 'No Heart Disease'}")
print(f"Probability: No Disease={rf_probability[0]:.4f}, Disease={rf_probability[1]:.4f}")

## 10. Conclusion

This notebook demonstrates:
- Loading and exploring heart disease dataset
- Data preprocessing and feature scaling
- Training multiple machine learning models (Logistic Regression and Random Forest)
- Model evaluation and comparison
- Making predictions on new data

The best performing model can be selected based on the evaluation metrics and used for heart disease prediction.